# Vietnamese News Feature Extraction With Scikit Learn

Turns the fully processed text from `260106_TextPreprocessingwithNLP` into TF-IDF features using
`sklearn.feature_extraction.text`, following the official guide at
https://scikit-learn.org/stable/modules/feature_extraction.html.

All text processing (cleaning, normalizing, tokenizing, stopword removal) happens in that other
project. This notebook does none of that itself: it loads the already-finished text and applies
vectorization techniques to it, nothing more.

## Plan

1. Load the exported, fully processed text (`data/processed_news.parquet`), the full corpus, no
   sampling.
2. Build two TF-IDF representations with `CountVectorizer` + `TfidfTransformer`: unigram
   (`ngram_range=(1, 1)`) and unigram plus bigram (`ngram_range=(1, 2)`).
3. For each, report vocabulary size and sparsity, and show a term-level breakdown table (term,
   raw Bag of Words count, IDF, TF-IDF) for one example document.

## Imports

In [1]:
import os

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

## Loading The Processed Data

This is the finished output of the preprocessing notebook: cleaned, Unicode-normalized, lowercased,
teencode-folded, tokenized, and stopwords already removed. No further text processing happens here.

In [2]:
df = pd.read_parquet(os.path.join("..", "data", "raw", "processed_news.parquet"))
print(df.shape)
df.head()

(184539, 4)


       id        source             topic  \
0  218270     docbao.vn         Pháp luật   
1  218269        vtc.vn      Sống kết nối   
2  218268  thanhnien.vn          Giáo dục   
3  218267     vnexpress          Thế giới   
4  218266          soha  Thời sự - Xã hội   

                                content_no_stopwords  
0  chiều công an tỉnh thừa thiên huế thông ban đầ...  
1  trưởng phát triển kỹ thuật truyền thông truyền...  
2  kết thi nghiệp thpt trung bình môn toán ngoại ...  
3  thống đốc kentucky andy beshear hôm đợt mưa lũ...  
4  vụ tai nạn giao thông liên hoàn phố đi tam bạc...  

## Building The TF-IDF Features

`TfidfVectorizer` is really `CountVectorizer` followed by `TfidfTransformer` (the sklearn guide
shows this equivalence directly). Using the two pieces separately here means the raw Bag of Words
counts, the IDF values, and the final TF-IDF weights are all available afterward to build the
breakdown table below, instead of only getting the final TF-IDF numbers.

Two configurations: unigram only (`ngram_range=(1, 1)`) and unigram plus bigram
(`ngram_range=(1, 2)`). `min_df=5` drops terms that appear in fewer than 5 documents; the sklearn
guide notes that adding bigrams roughly squares the vocabulary size, and most of the extra bigrams
are one-off noise.

In [3]:
def build_tfidf(texts, ngram_range, min_df=5):
    count_vectorizer = CountVectorizer(min_df=min_df, ngram_range=ngram_range)
    count_matrix = count_vectorizer.fit_transform(texts)

    tfidf_transformer = TfidfTransformer()
    tfidf_matrix = tfidf_transformer.fit_transform(count_matrix)

    return count_vectorizer, count_matrix, tfidf_transformer, tfidf_matrix


def breakdown_table(vectorizer, count_matrix, transformer, tfidf_matrix, row, top_n=15):
    table = pd.DataFrame({
        "Term": vectorizer.get_feature_names_out(),
        "BoW Count": count_matrix[row].toarray().ravel(),
        "IDF": transformer.idf_,
        "TF-IDF": tfidf_matrix[row].toarray().ravel(),
    })
    table = table[table["BoW Count"] > 0].sort_values("TF-IDF", ascending=False)
    return table.head(top_n)


corpus = df["content_no_stopwords"].tolist()
EXAMPLE_ROW = 0

### Unigram

In [4]:
cv_unigram, count_unigram, tfidf_tr_unigram, X_unigram = build_tfidf(corpus, ngram_range=(1, 1))

n_docs, n_features = count_unigram.shape
sparsity = 1 - count_unigram.nnz / (n_docs * n_features)
print(f"documents: {n_docs:,}  vocabulary: {n_features:,}  sparsity: {sparsity:.4%}")

breakdown_table(cv_unigram, count_unigram, tfidf_tr_unigram, X_unigram, EXAMPLE_ROW)

documents: 184,539  vocabulary: 32,701  sparsity: 99.5783%


,Term,BoW Count,IDF,TF-IDF
11969,huế,9,5.107655,0.422400
5121,chợ,5,4.551859,0.209131
897,an,10,2.224614,0.204416
28012,tiệm,4,5.533760,0.203395
6223,cướp,4,5.369426,0.197355
30451,vàng,6,3.576241,0.197169
26806,súng,4,4.990861,0.183441
29071,tượng,7,2.706858,0.174110
6182,công,13,1.436108,0.171550
27558,thiên,5,3.720372,0.170929


### Unigram + Bigram

In [5]:
cv_bigram, count_bigram, tfidf_tr_bigram, X_bigram = build_tfidf(corpus, ngram_range=(1, 2))

n_docs, n_features = count_bigram.shape
sparsity = 1 - count_bigram.nnz / (n_docs * n_features)
print(f"documents: {n_docs:,}  vocabulary: {n_features:,}  sparsity: {sparsity:.4%}")

breakdown_table(cv_bigram, count_bigram, tfidf_tr_bigram, X_bigram, EXAMPLE_ROW)

documents: 184,539  vocabulary: 886,893  sparsity: 99.9617%


,Term,BoW Count,IDF,TF-IDF
229303,huế,9,5.107655,0.238313
105849,công an,9,3.210996,0.149819
654073,tp huế,4,7.146736,0.148201
730601,tỉnh thừa,4,6.651731,0.137937
863398,đối tượng,7,3.681079,0.133585
87191,chợ đông,3,8.250424,0.128316
86916,chợ,5,4.551859,0.117990
592057,thiên huế,4,5.661685,0.117406
636325,thừa thiên,4,5.655397,0.117276
649048,tiệm vàng,3,7.490832,0.116503


## Summary

In [6]:
summary = pd.DataFrame([
    ("unigram", *X_unigram.shape, 1 - count_unigram.nnz / (count_unigram.shape[0] * count_unigram.shape[1])),
    ("unigram+bigram", *X_bigram.shape, 1 - count_bigram.nnz / (count_bigram.shape[0] * count_bigram.shape[1])),
], columns=["ngram_range", "documents", "vocabulary_size", "sparsity"])
summary

,ngram_range,documents,vocabulary_size,sparsity
0,unigram,184539,32701,0.995783
1,unigram+bigram,184539,886893,0.999617


## Next Steps

* Compare unigram versus unigram+bigram on an actual downstream task rather than by vocabulary
  size alone: fit a classifier (e.g. `LogisticRegression` or `MultinomialNB`) using the `topic`
  column as the label, on a held out split, for each of the two matrices above, and check accuracy.
* The preprocessing notebook tokenizes with NLTK's `word_tokenize`, which only splits on syllables,
  not real Vietnamese words. A proper Vietnamese word segmenter (`underthesea`) was evaluated for
  that step and found too slow for the full corpus (roughly 2 hours versus a few minutes); see
  `Personal Note.md` for the benchmark and the reasoning. If that tradeoff changes, it would be
  fixed in the preprocessing notebook, not here.